In [ ]:
'''
pip install langchain
pip install dotenv
pip install langchain-community
pip install langchain-google-genai
pip install google-search-results
pip install bs4
pip install requests
'''

In [ ]:
from typing import List, Dict
import requests
from bs4 import BeautifulSoup
import json
import pandas as pd
from serpapi import GoogleSearch
import os
import re
from dotenv import load_dotenv

In [ ]:
load_dotenv("secrets.env")
serp_key = os.getenv("SERPAPI_API_KEY")

In [229]:
event_categories = ['Business', 'Health', 'Music', 'Charity', 'Social Causes', 'Community','Culture', 'Education', 'Online',
                    'Science', 'Nightlife', 'Food & Drink', 'Outdoors', 'Performance', 'Tastings', 'Food Festival',
                    'Dating', 'LGBTQ', 'Comedy', 'Fitness', 'Tech', 'Home', 'Lifestyle', 'Parties', 'Film', 'Theater']

In [230]:
def search_eventbrite(query: str, location: str, max_results: int = 10) -> List[Dict]:
  
    search_query = f'site:eventbrite.com "{query}" "{location}"'
    
    params = {
        "engine": "google",
        "q": search_query,
        "api_key": serp_key,
        "num": max_results
    }
    
    search = GoogleSearch(params)
    results = search.get_dict()
    
    events = []
    for r in results.get("organic_results", [])[:max_results]:
        events.append({
            "url": r.get("link")
        })
    return events

In [231]:
results = {}

for category in event_categories:
    results[category] = {
        "category": category,
        "events": search_eventbrite(category, "New York", max_results=10)
    }

In [232]:
rows = []
for cat, details in results.items():
    category = details["category"]
    for event in details["events"]:
        rows.append({"category": category, "url": event["url"]})


df = pd.DataFrame(rows)
df["category"] = df["category"].replace("LGBTQ", "Community & Culture")
df["category"] = df["category"].replace("Community", "Community & Culture")
df["category"] = df["category"].replace("Culture", "Community & Culture")
df["category"] = df["category"].replace("Outdoors", "Health & Fitness")
df["category"] = df["category"].replace("Health", "Health & Fitness")
df["category"] = df["category"].replace("Fitness", "Health & Fitness")
df["category"] = df["category"].replace("Home", "Home & Lifestyle")
df["category"] = df["category"].replace("Lifestyle", "Home & Lifestyle")
df["category"] = df["category"].replace("Comedy", "Comedy & Performance")
df["category"] = df["category"].replace("Performance", "Comedy & Performance")
df["category"] = df["category"].replace("Charity", "Charity & Social Causes")
df["category"] = df["category"].replace("Social Causes", "Charity & Social Causes")
df["category"] = df["category"].replace("Science", "STEM")
df["category"] = df["category"].replace("Tech", "STEM")
df["category"] = df["category"].replace("Nightlife", "Nightlife & Parties")
df["category"] = df["category"].replace("Parties", "Nightlife & Parties")
df["category"] = df["category"].replace("Food Festival", "Food & Drink")
df["category"] = df["category"].replace("Tastings", "Food & Drink")
df["category"] = df["category"].replace("Film", "Comedy & Performance")
df["category"] = df["category"].replace("Theater", "Comedy & Performance")
df

,category,url
0,Business,https://www.eventbrite.com/b/ny--new-york/busi...
1,Business,https://www.eventbrite.com/d/ny--new-york/free...
2,Business,https://www.eventbrite.com/d/ny--new-york/busi...
3,Business,https://www.eventbrite.com/d/ny--new-york/busi...
4,Business,https://www.eventbrite.com/d/ny--new-york/busi...
...,...,...
249,Comedy & Performance,https://www.eventbrite.com/b/united-states--ne...
250,Comedy & Performance,https://www.eventbrite.com/o/new-york-public-l...
251,Comedy & Performance,https://www.eventbrite.com/d/ny--new-york/part...
252,Comedy & Performance,https://www.eventbrite.com/b/ny--yonkers/arts/...


In [233]:
df['category'].value_counts()

category
Comedy & Performance       40
Community & Culture        30
Health & Fitness           30
Food & Drink               30
Nightlife & Parties        20
STEM                       20
Home & Lifestyle           19
Charity & Social Causes    15
Business                   10
Music                      10
Online                     10
Education                  10
Dating                     10
Name: count, dtype: int64

In [260]:
def is_actual_event(url: str) -> bool:

    return re.search(r"-tickets-\d+$", url) is not None

def get_event_links(url: str, category: str, idx: int) -> pd.DataFrame:
    try:
        r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=10)
        r.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"⚠️ Skipping index {idx}, url={url} due to error: {e}")
        return pd.DataFrame(columns=["category", "url"])  # empty df if failed

    soup = BeautifulSoup(r.text, "html.parser")

    rows = []
    for a in soup.find_all("a", href=True):
        href = a["href"]
        rows.append({"category": category, "url": href})

    return pd.DataFrame(rows)


In [ ]:
expanded_rows = []
for idx, row in df.iterrows():
    expanded_rows.append(get_event_links(row["url"], row["category"], idx))

final_df = pd.concat([df] + expanded_rows, ignore_index=True)
final_df['actual'] = final_df['url'].apply(is_actual_event)
final_df = final_df[final_df['actual'] == True]
final_df 

⚠️ Skipping index 162, url=https://www.eventbrite.com/e/saturday-night-ages-24-36-speed-dating-in-new-york-cityspeed-nyc-tickets-1428426685669 due to error: 404 Client Error: Not Found for url: https://www.eventbrite.com/e/saturday-night-ages-24-36-speed-dating-in-new-york-cityspeed-nyc-tickets-1428426685669


In [278]:
final_df = final_df[final_df['actual'] == True]
final_df.reset_index()
final_df.drop_duplicates(subset='url', inplace=True)

In [ ]:
# Add a try catch block if website invalid

def get_event_metadata(url: str):
    try:
        session = requests.Session()
        session.headers.update({
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                        "AppleWebKit/537.36 (KHTML, like Gecko) "
                        "Chrome/120.0.0.0 Safari/537.36"
        })

        response = session.get(url)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"⚠️ Skipping index url={url} due to error: {e}")
        return pd.DataFrame(columns=["category", "url"])
    

    soup = BeautifulSoup(response.text, "html.parser")


    meta_tags = {meta.get("property") or meta.get("name"): meta.get("content")
                for meta in soup.find_all("meta") if meta.get("content")}

    keys_to_keep = [
        'og:title',
        'og:description',
        'og:url',
        'event:start_time',
        'event:end_time',
        'event:location:latitude',
        'event:location:longitude',  
        'og:image'
    ]   

    filtered_meta_tags = {k: v for k, v in meta_tags.items() if k in keys_to_keep}
    filtered_meta_tags
    match = re.search(r'-tickets-(\d+)', filtered_meta_tags['og:url'])
    if match:
        filtered_meta_tags['id'] = match.group(1)

    return filtered_meta_tags


In [ ]:
final_df['metadata'] = final_df['url'].apply(get_event_metadata)

⚠️ Skipping index 253, url=https://www.eventbrite.com/e/saturday-night-ages-24-36-speed-dating-in-new-york-cityspeed-nyc-tickets-1428426685669 due to error: 404 Client Error: Not Found for url: https://www.eventbrite.com/e/saturday-night-ages-24-36-speed-dating-in-new-york-cityspeed-nyc-tickets-1428426685669
⚠️ Skipping index 253, url=https://www.eventbrite.com/e/moca-talks-with-karen-hao-empire-of-ai-tickets-1602071221149 due to error: 404 Client Error: Not Found for url: https://www.eventbrite.com/e/moca-talks-with-karen-hao-empire-of-ai-tickets-1602071221149


In [ ]:
Education,https://www.eventbrite.com/e/2025-education-in-new-york-summit-tickets-1393156501599,True,"{'og:image': 'https://cdn.evbuc.com/images/1045054723/130664635387/1/logo.20250603-183951', 'og:title': '2025 Education in New York Summit', 'og:description': 'Shaping Tomorrow’s Schools Through Innovation', 'og:url': 'https://www.eventbrite.com/e/2025-education-in-new-york-summit-tickets-1393156501599', 'event:location:latitude': '40.7059752', 'event:location:longitude': '-74.01857889999997', 'event:start_time': '2025-08-14T09:00:00-04:00', 'event:end_time': '2025-08-14T15:30:00-04:00'}"